# CRNN+CTC — распознавание произвольного текста на номерных знаках

**Цель:** свой нейросетевой OCR, который читает **любой формат и шрифт** — стандартные UA, дипломатические, спецтехнику, такси, военные, **именные/vanity**, иностранные.

**Почему не CharCNN (ДЗ14) сам по себе:** CharCNN — per-slot классификатор, работает только при фиксированной длине/формате. Для именных номеров и нестандартных форматов нужна **sequence-модель**.

**Архитектура (Shi et al. 2015):**
- **CNN-backbone** в стиле ДЗ14 (Conv→BN→ReLU блоки, MaxPool) — извлекает признаки по ширине
- **2× BiLSTM** — моделирует последовательность символов
- **Linear + log_softmax** над `|charset|+1` классами (blank для CTC)
- **CTC-loss** — обучение без явного выравнивания символов

**Данные:**
- `data/synthetic_ocr/` — наш генератор (все форматы + именные + разные шрифты), сгенерить через `scripts/generate_synthetic_ocr.py`
- `data/ocr_crops/` — реальные кропы AUTO.RIA (если в VIA JSON есть GT-текст), сгенерить через `scripts/prepare_ocr_crops.py`

**Связь с ДЗ:** ДЗ13/14 (CNN + training loop), ДЗ15 (борьба с overfitting, метрики), ДЗ12 (EDA распределений).

In [ ]:
import sys, csv, random
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import cv2, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.crnn import CRNN, NUM_CLASSES, encode, decode_greedy, preprocess_for_crnn, ctc_collate, CHARSET

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| classes (incl. blank):', NUM_CLASSES, '| charset len:', len(CHARSET))

SYNTH = Path('../data/synthetic_ocr')
REAL = Path('../data/ocr_crops')

## 1. Dataset — читает synthetic + real одновременно

In [ ]:
class PlateOCRDataset(Dataset):
    """Читает пары (crop, text) из CSV-разметки. Возвращает (tensor 1×32×W, text)."""
    def __init__(self, sources, train=False, min_len=2, max_len=10):
        self.samples = []
        for root, csv_path in sources:
            if not csv_path.exists():
                print(f'[skip] {csv_path} не найден')
                continue
            with open(csv_path, encoding='utf-8') as f:
                for row in csv.DictReader(f):
                    text = row['text'].strip().upper()
                    if not (min_len <= len(text) <= max_len):
                        continue
                    if any(c not in CHARSET for c in text):
                        continue
                    self.samples.append((root / row['file'], text))
        self.train = train
        print(f'  собрано {len(self.samples)} семплов')

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, text = self.samples[i]
        img = cv2.imread(str(path))
        if img is None:
            return torch.zeros(1, 32, 64), text
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.train and random.random() < 0.3:
            rgb = self._jitter(rgb)
        tensor = preprocess_for_crnn(rgb, target_h=32, max_w=256)
        return tensor.squeeze(0), text

    @staticmethod
    def _jitter(img):
        alpha = random.uniform(0.8, 1.2); beta = random.uniform(-20, 20)
        return np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)


train_sources = [(SYNTH/'images/train', SYNTH/'labels_train.csv'), (REAL/'train', REAL/'labels_train.csv')]
val_sources   = [(SYNTH/'images/val',   SYNTH/'labels_val.csv'),   (REAL/'val',   REAL/'labels_val.csv')]

print('train:'); train_ds = PlateOCRDataset(train_sources, train=True)
print('val:');   val_ds   = PlateOCRDataset(val_sources,   train=False)

assert len(train_ds) > 0, 'нет train-данных — запустите scripts/generate_synthetic_ocr.py'

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0, collate_fn=ctc_collate)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0, collate_fn=ctc_collate)

## 2. EDA — длины строк и распределение видов номеров

In [ ]:
lengths = [len(t) for _, t in train_ds.samples]
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].hist(lengths, bins=range(1, 12)); ax[0].set_title('длины строк (train)'); ax[0].set_xlabel('chars')
# несколько примеров
samples = random.sample(train_ds.samples, min(8, len(train_ds.samples)))
ax[1].axis('off'); ax[1].set_title('случайные примеры')
grid = []
for p, t in samples:
    img = cv2.imread(str(p))
    if img is None: continue
    img = cv2.resize(img, (128, 32))
    grid.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
if grid: ax[1].imshow(np.vstack(grid))
plt.tight_layout(); plt.show()

## 3. Модель + CTC loss + оптимайзер

In [ ]:
model = CRNN(num_classes=NUM_CLASSES).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)

n_params = sum(p.numel() for p in model.parameters())
print(f'CRNN параметров: {n_params/1e6:.2f}M')

## 4. Train loop + CER на валидации

In [ ]:
def levenshtein(a, b):
    if len(a) < len(b): a, b = b, a
    if not b: return len(a)
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0]*len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(cur[j-1]+1, prev[j]+1, prev[j-1]+(ca != cb))
        prev = cur
    return prev[-1]

def evaluate(model, dl):
    model.eval(); total_cer = 0; total_chars = 0; exact = 0; n = 0
    with torch.no_grad():
        for imgs, _, _, texts in dl:
            imgs = imgs.to(DEVICE)
            log_probs = model(imgs)
            preds = decode_greedy(log_probs)
            for p, t in zip(preds, texts):
                total_cer += levenshtein(p, t)
                total_chars += max(1, len(t))
                exact += int(p == t)
                n += 1
    return total_cer/total_chars, exact/n

EPOCHS = 20
history = {'train_loss': [], 'val_cer': [], 'val_exact': []}
best_cer = 1.0

for epoch in range(1, EPOCHS+1):
    model.train(); total = 0; n = 0
    pbar = tqdm(train_dl, desc=f'ep {epoch:02d}', leave=False)
    for imgs, targets, target_lens, _ in pbar:
        imgs = imgs.to(DEVICE); targets = targets.to(DEVICE); target_lens = target_lens.to(DEVICE)
        log_probs = model(imgs)                    # (T, B, C)
        T = log_probs.size(0); B = log_probs.size(1)
        input_lens = torch.full((B,), T, dtype=torch.long, device=DEVICE)
        loss = ctc_loss(log_probs, targets, input_lens, target_lens)
        if not torch.isfinite(loss): continue
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        total += loss.item() * B; n += B
        pbar.set_postfix(loss=f'{loss.item():.3f}')
    tr_loss = total / max(n, 1)
    sched.step()

    val_cer, val_exact = evaluate(model, val_dl)
    history['train_loss'].append(tr_loss)
    history['val_cer'].append(val_cer)
    history['val_exact'].append(val_exact)
    print(f'ep {epoch:02d}  train_loss={tr_loss:.3f}  val_CER={val_cer:.3f}  val_exact={val_exact:.3f}')

    if val_cer < best_cer:
        best_cer = val_cer
        out = Path('../models/crnn_ocr.pt')
        out.parent.mkdir(exist_ok=True)
        torch.save({'model_state_dict': model.state_dict(), 'charset': CHARSET}, out)
        print(f'  ↳ saved best → {out}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot(history['train_loss']); ax[0].set_title('train CTC loss')
ax[1].plot(history['val_cer'], label='CER'); ax[1].plot(history['val_exact'], label='exact')
ax[1].set_title('val metrics'); ax[1].legend(); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 5. Инференс-демо на случайных val-семплах

In [ ]:
model.eval()
samples = random.sample(val_ds.samples, min(10, len(val_ds.samples)))
fig, axes = plt.subplots(len(samples), 1, figsize=(8, len(samples)*1.2))
with torch.no_grad():
    for ax, (p, gt) in zip(axes, samples):
        img = cv2.imread(str(p))
        if img is None: continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = preprocess_for_crnn(rgb).to(DEVICE)
        log_probs = model(tensor)
        pred = decode_greedy(log_probs)[0]
        ax.imshow(rgb); ax.axis('off')
        color = 'green' if pred == gt else 'red'
        ax.set_title(f'GT: {gt}    PRED: {pred}', color=color, fontsize=10)
plt.tight_layout(); plt.show()

## 6. Per-kind breakdown — какие форматы сеть читает лучше

In [ ]:
# kind зашит в имя файла синтетики: `0000123_named.png`
import pandas as pd
def kind_of(path: Path):
    parts = path.stem.split('_')
    return parts[-1] if len(parts) > 1 else 'real'

rows = []
model.eval()
with torch.no_grad():
    for imgs, _, _, texts in val_dl:
        imgs = imgs.to(DEVICE)
        preds = decode_greedy(model(imgs))
        rows.extend(zip(preds, texts))

paths = [p for p, _ in val_ds.samples]
kinds = [kind_of(p) for p in paths[:len(rows)]]
df = pd.DataFrame({'pred': [r[0] for r in rows], 'gt': [r[1] for r in rows], 'kind': kinds[:len(rows)]})
df['cer'] = df.apply(lambda r: levenshtein(r['pred'], r['gt']) / max(len(r['gt']), 1), axis=1)
df['exact'] = (df['pred'] == df['gt']).astype(int)
summary = df.groupby('kind').agg(n=('gt','count'), mean_CER=('cer','mean'), exact_pct=('exact',lambda x: 100*x.mean())).round(3)
summary

## Выводы

**Ключевое отличие от CharCNN (ДЗ14):** CTC убирает требование на фиксированную длину — сеть читает `AA1234BB`, `SANDY`, `001CD123` одной моделью.

**Что унаследовано из курса:**
- Backbone — Conv→BN→ReLU блоки и MaxPool (ДЗ14)
- Cosine LR + gradient clipping — стандартные приёмы борьбы с overfitting (ДЗ15)
- EDA распределений классов и per-kind метрики (ДЗ12, ДЗ15)

**Что добавлено сверх курса (обязательно для задачи):**
- BiLSTM (sequence-моделирование)
- CTC loss (alignment-free обучение)
- Мульти-шрифтовая синтетика для независимости от шрифта

Веса сохранены в `models/crnn_ocr.pt` — подхватываются `ALPRPipeline` при `cfg.ocr_backend='crnn'`.